## 1. Importing Dataset


In [23]:
import pandas as pd
df=pd.read_csv("C:/code/IBM Training/Project/Real estate Prediction/Data/Raw/historical_scheme_data.csv")


df.info()
print (df.shape)

<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Scheme_ID                 800 non-null    str    
 1   Location                  800 non-null    str    
 2   Plot_Size_SqFt            800 non-null    int64  
 3   Price_INR                 800 non-null    int64  
 4   Monthly_EMI_INR           800 non-null    int64  
 5   Park                      800 non-null    str    
 6   Clubhouse                 800 non-null    str    
 7   Distance_from_Metro_km    800 non-null    float64
 8   Advertising_Budget_INR    800 non-null    int64  
 9   Expected_Leads            800 non-null    int64  
 10  Expected_Bookings         800 non-null    int64  
 11  Estimated_Conversion_Pct  800 non-null    float64
 12  Demand_Level              800 non-null    str    
 13  Scheme_Success_Score      800 non-null    int64  
 14  Target_Age_Group     

## 2. Checking for duplicates And Missing Values

In [24]:

print("Dataset Shape:", df.shape)

print("\n--- Missing Values ---")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found.")

print("\nTotal Missing Values:", df.isnull().sum().sum())

print("\n--- Duplicate Rows ---")
print("Duplicate rows:", df.duplicated().sum())

print("\n--- Duplicate Scheme IDs ---")
print("Duplicate Scheme_IDs:", df["Scheme_ID"].duplicated().sum())

Dataset Shape: (800, 16)

--- Missing Values ---
No missing values found.

Total Missing Values: 0

--- Duplicate Rows ---
Duplicate rows: 0

--- Duplicate Scheme IDs ---
Duplicate Scheme_IDs: 0


## 3. Categorical & NUmerical Data Inspection

In [25]:


cat_cols = df.select_dtypes(include=["object", "string", "category"]).columns
num_cols = df.select_dtypes(include=["int64", "float64"]).columns

print("Categorical Columns:")
print(list(cat_cols))

for col in cat_cols:
    print(f"\n--- {col} ---")
    print("Unique values:", df[col].nunique())
    print(df[col].value_counts(dropna=False).head(20))


print("\n\nNumerical Columns:")
print(list(num_cols))

print("\n--- Numerical Statistics ---")
print(df[num_cols].describe().T)

Categorical Columns:
['Scheme_ID', 'Location', 'Park', 'Clubhouse', 'Demand_Level', 'Target_Age_Group', 'Most_Likely_Buyers']

--- Scheme_ID ---
Unique values: 800
Scheme_ID
SCH_0101    1
SCH_0102    1
SCH_0103    1
SCH_0104    1
SCH_0105    1
SCH_0106    1
SCH_0107    1
SCH_0108    1
SCH_0109    1
SCH_0110    1
SCH_0111    1
SCH_0112    1
SCH_0113    1
SCH_0114    1
SCH_0115    1
SCH_0116    1
SCH_0117    1
SCH_0118    1
SCH_0119    1
SCH_0120    1
Name: count, dtype: int64

--- Location ---
Unique values: 12
Location
Faridabad (Neharpar)           76
Gurugram (SPR/Sohna)           75
Greater Noida                  75
Pune (Hinjewadi/Wakad)         70
Ahmedabad (SG Highway)         68
Delhi (Outer/Bawana)           66
Kolkata (New Town/Rajarhat)    64
Bengaluru (North)              64
Chennai (OMR/Guduvanchery)     64
Pune (Wagholi/Kharadi)         60
Bengaluru (East)               60
Hyderabad (Tellapur/Kollur)    58
Name: count, dtype: int64

--- Park ---
Unique values: 2
Park
No   

 ## 4. POTENTIAL FEATURES & DOWNSTREAM VALIDATION

In [26]:


print("--- Potential Feature Relationships ---")

# Correlation between numerical variables
print("\nNumerical Correlation Matrix:")
print(df[num_cols].corr().round(3))


# Business-logic validation
print("\n--- Business Logic Checks ---")

# Bookings should not exceed leads
print(
    "Bookings > Leads:",
    (df["Expected_Bookings"] > df["Expected_Leads"]).sum()
)

# Calculate conversion rate independently
calculated_conversion = (
    df["Expected_Bookings"] /
    df["Expected_Leads"]
) * 100

conversion_difference = (
    calculated_conversion -
    df["Estimated_Conversion_Pct"]
).abs()

print(
    "Conversion mismatch > 1%:",
    (conversion_difference > 1).sum()
)

# Potential modelling target
print("\n--- Potential Target ---")
print("Scheme_Success_Score")

print("\nPotential numerical predictors:")
print([
    col for col in num_cols
    if col != "Scheme_Success_Score"
])

--- Potential Feature Relationships ---

Numerical Correlation Matrix:
                          Plot_Size_SqFt  Price_INR  Monthly_EMI_INR  \
Plot_Size_SqFt                     1.000      0.763            0.763   
Price_INR                          0.763      1.000            1.000   
Monthly_EMI_INR                    0.763      1.000            1.000   
Distance_from_Metro_km            -0.039     -0.031           -0.031   
Advertising_Budget_INR            -0.060     -0.053           -0.054   
Expected_Leads                    -0.040     -0.034           -0.034   
Expected_Bookings                 -0.026     -0.042           -0.042   
Estimated_Conversion_Pct           0.031     -0.059           -0.059   
Scheme_Success_Score               0.019     -0.029           -0.029   

                          Distance_from_Metro_km  Advertising_Budget_INR  \
Plot_Size_SqFt                            -0.039                  -0.060   
Price_INR                                 -0.031        

## 5. FINAL CLEAN DATASET CHECK

In [27]:

checks = {
    "Rows": len(df),
    "Columns": len(df.columns),
    "Missing Values": df.isnull().sum().sum(),
    "Duplicate Rows": df.duplicated().sum(),
    "Duplicate Scheme IDs": df["Scheme_ID"].duplicated().sum(),
    "Invalid Plot Size": (df["Plot_Size_SqFt"] <= 0).sum(),
    "Invalid Price": (df["Price_INR"] <= 0).sum(),
    "Invalid EMI": (df["Monthly_EMI_INR"] <= 0).sum(),
    "Negative Metro Distance": (df["Distance_from_Metro_km"] < 0).sum(),
    "Negative Advertising Budget": (df["Advertising_Budget_INR"] < 0).sum(),
    "Negative Leads": (df["Expected_Leads"] < 0).sum(),
    "Negative Bookings": (df["Expected_Bookings"] < 0).sum(),
    "Invalid Conversion %": (
        (df["Estimated_Conversion_Pct"] < 0) |
        (df["Estimated_Conversion_Pct"] > 100)
    ).sum(),
    "Invalid Success Score": (
        (df["Scheme_Success_Score"] < 0) |
        (df["Scheme_Success_Score"] > 100)
    ).sum()
}

for parameter, result in checks.items():
    print(f"{parameter:<30}: {result}")

print("\nFinal Shape:", df.shape)
print("\nDataset ready for EDA / Feature Engineering / Modelling.")

Rows                          : 800
Columns                       : 16
Missing Values                : 0
Duplicate Rows                : 0
Duplicate Scheme IDs          : 0
Invalid Plot Size             : 0
Invalid Price                 : 0
Invalid EMI                   : 0
Negative Metro Distance       : 0
Negative Advertising Budget   : 0
Negative Leads                : 0
Negative Bookings             : 0
Invalid Conversion %          : 0
Invalid Success Score         : 0

Final Shape: (800, 16)

Dataset ready for EDA / Feature Engineering / Modelling.
